In [7]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Re-establishthe database connection engine
DB_USER = "postgres"
DB_PASSWORD = "Postgres"  
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "bank_reviews"

connection_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_url)

# 2. Define the static bank metadata to populate the parent 'banks' table

bank_metadata = pd.DataFrame([
    {'bank_id': 'cbe', 'bank_name': 'Commercial Bank of Ethiopia', 'app_name': 'CBE Birr / Mobile Banking'},
    {'bank_id': 'boa', 'bank_name': 'Bank of Abyssinia', 'app_name': 'Apollo / BoA Mobile Wallet'},
    {'bank_id': 'dashen', 'bank_name': 'Dashen Bank', 'app_name': 'Amole / Dashen Bank app'}
])

# 3. Prepare the reviews data from your 'df_processed' DataFrame

# Define the path to the processed data file
processed_data_path = r'C:\Users\nemsa\KAIM Challenge Project\Fintech-Review-Analytics\data\processed\fintech_sentiment_analysis_resutls.csv'

df_loaded = pd.read_csv(processed_data_path)

df_to_insert = df_loaded.copy()

# Standardize the bank column labels to match our database parent IDs ('cbe', 'boa', 'dashen')
df_to_insert['bank_id'] = df_to_insert['bank'].str.lower().str.replace('bank of abyssinia', 'boa')

# Re-map your pandas columns into a new DataFrame that exactly matches our SQL table schema
reviews_sql_table = pd.DataFrame({
    'review_id': df_to_insert['review_id'],
    'bank_id': df_to_insert['bank_id'],
    'review_text': df_to_insert['review_text'],
    'rating': df_to_insert['rating'].astype(int),
    'sentiment_label': df_to_insert['sentiment_label'],
    'sentiment_score': df_to_insert['sentiment_score'],
    'identified_theme': df_to_insert['identified_theme'],
    'source': 'Google Play Store'
})

# Filter out any 'unknown' banks to prevent Foreign Key errors
reviews_sql_table = reviews_sql_table[reviews_sql_table['bank_id'] != 'unknown']

# 4. Stream the data straight into PostgreSQL
try:
    with engine.begin() as connection:
        print("🗄️ Populating parent table 'banks' with metadata framework...")
        for _, row in bank_metadata.iterrows():
            connection.execute(text("""
                INSERT INTO banks (bank_id, bank_name, app_name) 
                VALUES (:bank_id, :bank_name, :app_name)
                ON CONFLICT (bank_id) DO NOTHING;
            """), dict(row))
            
        print(f"📥 Bulk-streaming {len(reviews_sql_table)} records (excluding date)...")
        # to_sql handles the thousands of insert statements automatically and blazingly fast
        reviews_sql_table.to_sql('reviews', con=connection, if_exists='append', index=False)
        
    print("\n🚀 SUCCESS! All data has been securely saved to PostgreSQL!")

except Exception as e:
    print(f"\n❌ Ingestion failed. Database transaction rolled back safely. Error: {e}")

🗄️ Populating parent table 'banks' with metadata framework...
📥 Bulk-streaming 1200 records (excluding date)...

❌ Ingestion failed. Database transaction rolled back safely. Error: (psycopg2.errors.NotNullViolation) null value in column "review_date" of relation "reviews" violates not-null constraint
DETAIL:  Failing row contains (REV_0000, commercial bank of ethiopia, very very good 👍 thanks commercial bank of ethiopia 🤩😘..., 5, null, POSITIVE, 0.8470, General/Other, Google Play Store).

[SQL: INSERT INTO reviews (review_id, bank_id, review_text, rating, sentiment_label, sentiment_score, identified_theme, source) VALUES (%(review_id__0)s, %(bank_id__0)s, %(review_text__0)s, %(rating__0)s, %(sentiment_label__0)s, %(sentiment_score__0)s, %(i ... 173897 characters truncated ... 9)s, %(sentiment_label__999)s, %(sentiment_score__999)s, %(identified_theme__999)s, %(source__999)s)]
[parameters: {'rating__0': 5, 'sentiment_label__0': 'POSITIVE', 'identified_theme__0': 'General/Other', 'review